# Link Prediction with PyTorch Geometric
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/08_Graphs_Networks/graph_link_prediction_pyg.ipynb)

Node classification asks 'what is this node?'; link prediction asks 'which connections are missing?' - the engine behind friend suggestions, 'people also bought', and drug interaction discovery.

Recipe: encode nodes with a GCN -> score candidate edges by dot product of endpoint embeddings -> train against real vs sampled fake edges.

In [ ]:
!pip install -q torch_geometric scikit-learn

## 1. Cora + automatic edge splitting

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.transforms import RandomLinkSplit

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

splitter = RandomLinkSplit(split_labels=True, num_val=0.1, num_test=0.1,
                           is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = splitter(data)
print("train pos edges:", train_data.edge_label_index.size(1) // 2,
      "| test labels:", test_data.edge_label_index.size(1))

`edge_label_index` holds positive AND sampled-negative edges; `edge_label` is 1/0 ground truth.

## 2. Encoder + decoder

In [ ]:
class LinkPredictor(torch.nn.Module):
    def __init__(self, hidden=64, out=32):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_features, hidden)
        self.conv2 = GCNConv(hidden, out)

    def encode(self, x, edge_index):
        h = self.conv1(x, edge_index).relu()
        return self.conv2(h, edge_index)

    def decode(self, z, edge_label_index):
        src, dst = edge_label_index
        return (z[src] * z[dst]).sum(dim=-1)       # higher = more likely link

model = LinkPredictor()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

## 3. Train / validate / test

In [ ]:
def epoch_step(data, train=True):
    model.train(train)
    if train:
        opt.zero_grad()
    z = model.encode(data.x, data.edge_index)
    logits = model.decode(z, data.edge_label_index)
    loss = F.binary_cross_entropy_with_logits(logits, data.edge_label)
    if train:
        loss.backward(); opt.step()
    return loss.item()

from sklearn.metrics import roc_auc_score

@torch.no_grad()
def auc(data):
    model.eval()
    z = model.encode(data.x, data.edge_index)
    scores = model.decode(z, data.edge_label_index).sigmoid().numpy()
    return roc_auc_score(data.edge_label.numpy(), scores)

for ep in range(1, 201):
    loss = epoch_step(train_data)
    if ep % 50 == 0:
        print(f"epoch {ep:3d}  loss={loss:.4f}  val AUC={auc(val_data):.4f}")
print(f"TEST AUC: {auc(test_data):.4f}")

## 4. Rank the most likely MISSING citations

In [ ]:
@torch.no_grad()
def top_missing(k=8):
    model.eval()
    z = model.encode(test_data.x, test_data.edge_index)
    src, dst = test_data.edge_label_index
    pos = test_data.edge_label.bool()
    cand = [(int(s), int(d)) for s, d, p in zip(src, dst, pos) if not p]
    scores = model.decode(z, torch.tensor([c[0] for c in cand] +
                                          [c[1] for c in cand]).view(2, -1)).sigmoid()
    ranked = sorted(zip(cand, scores.tolist()), key=lambda x: -x[1])
    return ranked[:k]

for (a, b), s in top_missing():
    print(f"paper {a} <-> paper {b}   p={s:.3f}")

## Takeaways
- Same encoder as node classification; only the DECODER changes per task.
- Negative sampling quality decides everything - random negatives are easy; hard negatives (shared neighbors) make robust models.
- Extensions: `NetConv`-style heterogeneous graphs, temporal link prediction, knowledge-graph completion (TransE/RotatE).
- Production framing: candidate generation (fast heuristic top-1000) -> learned reranker (this model).